# ENVS 117 Week 3 Thursday: In-class Exercises
Work in pairs, about 10 minutes each. Run the setup cells first.

## Setup

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

BASE = ("https://github.com/opengeos/"
        "datasets/releases/download/")
files = ["us/us_counties.geojson",
         "us/cities.csv",
         "raster/dem_90m.tif"]

Path("data").mkdir(exist_ok=True)
for f in files:
    name = f.split("/")[-1]
    if not Path("data/" + name).exists():      # skip if already downloaded
        urlretrieve(BASE + f, "data/" + name)
        print("downloaded", name)
    else:
        print("already have", name)

already have us_counties.geojson
already have cities.csv
already have dem_90m.tif


In [2]:
import os
import math
import pandas as pd
import geopandas as gpd
import rasterio

## T1. California counties, two formats

**1.** Read `data/us_counties.geojson` with `gpd.read_file`.

In [3]:
# your code here
counties = gpd.read_file("data/us_counties.geojson")

**2.** Print its type, shape, column names and CRS.

In [4]:
# your code here
print(type(counties))
print(counties.shape)
print(counties.columns)
print(counties.crs)

<class 'geopandas.geodataframe.GeoDataFrame'>
(3221, 7)
Index(['GEO_ID', 'STATE', 'COUNTY', 'NAME', 'LSAD', 'CENSUSAREA', 'geometry'], dtype='str')
EPSG:4269


**3.** Keep California: `STATE` equals `"06"`. How many counties?

In [5]:
# your code here
ca_counties = counties[counties["STATE"].astype(str).str.zfill(2) == "06"].copy()
print("California county count:", len(ca_counties))

California county count: 58


**4.** What geometry type are they? What are the `total_bounds`?

In [6]:
# your code here
print("Geometry types:", ca_counties.geom_type.value_counts().to_dict())
print("Total bounds (minx, miny, maxx, maxy):", ca_counties.total_bounds)

Geometry types: {'MultiPolygon': 58}
Total bounds (minx, miny, maxx, maxy): [-124.409591     32.534156   -114.13442654   42.009518  ]


**5.** Save California as `data/ca_counties.gpkg` and as `data/ca_counties.shp`.

In [7]:
# your code here
ca_counties.to_file("data/ca_counties.gpkg", driver="GPKG")
ca_counties.to_file("data/ca_counties.shp", driver="ESRI Shapefile")
print("Saved GeoPackage and Shapefile.")

Saved GeoPackage and Shapefile.


**6.** List the data folder with `os.listdir`. How many files did each format create?

In [8]:
# your code here
data_files = sorted(os.listdir("data"))
print("Files in data folder:", data_files)
print("GeoPackage files:", [f for f in data_files if f == "ca_counties.gpkg"])
shapefile_parts = [f for f in data_files if f.startswith("ca_counties.") and f != "ca_counties.gpkg"]
print("Shapefile component files:", shapefile_parts)
print("Number of files created by GeoPackage:", 1)
print("Number of files created by Shapefile:", len(shapefile_parts))

Files in data folder: ['.ipynb_checkpoints', 'bay_area_pm25.csv', 'bay_area_pm25.gpkg', 'ca_counties.cpg', 'ca_counties.dbf', 'ca_counties.gpkg', 'ca_counties.prj', 'ca_counties.shp', 'ca_counties.shx', 'cities.csv', 'creek_sites.csv', 'data', 'dem_90m.tif', 'us_counties.geojson', 'us_states.geojson']
GeoPackage files: ['ca_counties.gpkg']
Shapefile component files: ['ca_counties.cpg', 'ca_counties.dbf', 'ca_counties.prj', 'ca_counties.shp', 'ca_counties.shx']
Number of files created by GeoPackage: 1
Number of files created by Shapefile: 5


**Discuss:** Why is STATE stored as the text "06" and not the number 6? Which format would you email to a classmate?

## T2. From a CSV of cities to points

**1.** Read `data/cities.csv` with `pd.read_csv`. Which columns hold the coordinates?

In [10]:
# your code here
cities = pd.read_csv("data/cities.csv")
print("Columns:", cities.columns.tolist())
cols = {c.lower(): c for c in cities.columns}
lat_col = next((cols[k] for k in ["latitude", "lat", "y"] if k in cols), None)
lon_col = next((cols[k] for k in ["longitude", "lon", "long", "lng", "x"] if k in cols), None)
if lat_col is None or lon_col is None:
    raise ValueError("Could not identify latitude/longitude columns; inspect the printed columns.")
print("Latitude column:", lat_col)
print("Longitude column:", lon_col)

Columns: ['name', 'sov_a3', 'latitude', 'longitude', 'pop_max', 'region']
Latitude column: latitude
Longitude column: longitude


**2.** Build a GeoDataFrame with `gpd.points_from_xy` and `crs="EPSG:4326"`. (x first, and x is longitude.)

In [11]:
# your code here
cities_gdf = gpd.GeoDataFrame(
    cities.copy(),
    geometry=gpd.points_from_xy(cities[lon_col], cities[lat_col]),
    crs="EPSG:4326"
)
cities_gdf.head()

,name,sov_a3,latitude,longitude,pop_max,region,geometry
0,San Bernardino,USA,34.12038,-117.30003,1745000,West,POINT (-117.30003 34.12038)
1,Bridgeport,USA,41.17998,-73.19996,1018000,Northeast,POINT (-73.19996 41.17998)
2,Rochester,USA,43.17043,-77.61995,755000,Northeast,POINT (-77.61995 43.17043)
3,St. Paul,USA,44.94399,-93.08497,734854,Midwest,POINT (-93.08497 44.94399)
4,Billings,USA,45.78830,-108.54000,104552,West,POINT (-108.54 45.7883)


**3.** Is its CRS the same as the counties' CRS? Print the comparison.

In [12]:
# your code here
print("Cities CRS:", cities_gdf.crs)
print("Counties CRS:", counties.crs)
print("Same CRS?", cities_gdf.crs == counties.crs)

Cities CRS: EPSG:4326
Counties CRS: EPSG:4269
Same CRS? False


**4.** Convert the points to the counties' CRS with `to_crs`.

In [14]:
# your code here
cities_projected = cities_gdf.to_crs(counties.crs)
print("Converted cities CRS:", cities_projected.crs)

Converted cities CRS: EPSG:4269


**5.** Pick "California" cities with a box: latitude 32.5 to 42.0 and longitude -124.5 to -114.1. Use `.between()`.

In [18]:
# your code here
lat = pd.to_numeric(cities_gdf[lat_col], errors="coerce")
lon = pd.to_numeric(cities_gdf[lon_col], errors="coerce")
ca_box = cities_gdf[lat.between(32.5, 42.0) & lon.between(-124.5, -114.1)].copy()
print("Cities inside the California bounding box:", len(ca_box))
name_col = next((c for c in cities.columns if c.lower() in ["city", "name", "city_name", "place"]), None)
state_col = next((c for c in cities.columns if c.lower() in ["state", "state_name", "state_code", "province"]), None)
show_cols = [c for c in [name_col, state_col, lat_col, lon_col] if c is not None]
print(ca_box[show_cols].to_string(index=False))
if state_col is not None:
    state_text = ca_box[state_col].astype(str).str.strip().str.lower()
    outside_ca = ca_box[~state_text.isin(["ca", "california"])]
    print("Cities in the box whose state field is not California/CA:")
    print(outside_ca[show_cols].to_string(index=False))
else:
    print("No state column found, so identify the three out-of-state cities by checking the printed city names and coordinates against a map.")

Cities inside the California bounding box: 12
          name  latitude  longitude
San Bernardino  34.12038 -117.30003
 Santa Barbara  34.43499 -119.71999
        Fresno  36.74772 -119.77298
        Eureka  40.80222 -124.14750
          Reno  39.52998 -119.82001
          Elko  40.83251 -115.76199
      San Jose  37.29998 -121.84999
    Sacramento  38.57502 -121.47004
     Las Vegas  36.21000 -115.22001
     San Diego  32.82002 -117.17999
 San Francisco  37.74001 -122.45998
   Los Angeles  33.98998 -118.17998
No state column found, so identify the three out-of-state cities by checking the printed city names and coordinates against a map.


**Look closely:** three cities in your box are not in California. Which ones, and why did the box catch them?

## T3. The Sierra Nevada elevation model

**1-2.** Open `data/dem_90m.tif` with rasterio inside a `with` block. Print crs, width, height, count, dtypes, res, nodata and bounds.

**3.** Read band 1 into `dem`. Print its shape and type.

In [19]:
# your code here
with rasterio.open("data/dem_90m.tif") as src:
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Band count:", src.count)
    print("Data types:", src.dtypes)
    print("Resolution:", src.res)
    print("NoData value:", src.nodata)
    print("Bounds:", src.bounds)
    dem = src.read(1, masked=True)
print("DEM shape:", dem.shape)
print("DEM type:", type(dem))
print("DEM data dtype:", dem.dtype)

CRS: EPSG:3857
Width: 4269
Height: 3113
Band count: 1
Data types: ('int16',)
Resolution: (90.0, 89.99579177642138)
NoData value: None
Bounds: BoundingBox(left=-13442488.3428, bottom=4388214.6777, right=-13058278.3428, top=4668371.5775)
DEM shape: (3113, 4269)
DEM type: <class 'numpy.ma.MaskedArray'>
DEM data dtype: int16


**4.** Find the lowest, highest and mean elevation, rounded to 1 decimal.

In [20]:
# your code here
valid_dem = dem.compressed()  # excludes masked/no-data cells
print("Lowest elevation (m):", round(float(valid_dem.min()), 1))
print("Highest elevation (m):", round(float(valid_dem.max()), 1))
print("Mean elevation (m):", round(float(valid_dem.mean()), 1))

Lowest elevation (m): 15.0
Highest elevation (m): 4338.0
Mean elevation (m): 1582.3


**5.** How many cells are above 3,000 m? Hint: `(dem > 3000).sum()`

In [21]:
# your code here
print("Cells above 3,000 m:", int(((dem > 3000).filled(False)).sum()))

Cells above 3,000 m: 859074


**Think:** the resolution says 90 m, but the CRS is Web Mercator. At 37.6 N, how wide is one cell on the ground?

In [22]:
# your code here
latitude = 37.6
map_cell_width_m = 90
print("Approximate ground width at 37.6° N (m):", round(map_cell_width_m * math.cos(math.radians(latitude)), 1))

Approximate ground width at 37.6° N (m): 71.3


## T4. Santa Clara County, three areas

**1.** From your California counties, select `NAME` equal to `"Santa Clara"`.

In [23]:
# your code here
santa_clara = ca_counties[ca_counties["NAME"] == "Santa Clara"].copy()
print("Matching rows:", len(santa_clara))
santa_clara

Matching rows: 1


,GEO_ID,STATE,COUNTY,NAME,LSAD,CENSUSAREA,geometry
235,0500000US06085,06,085,Santa Clara,County,1290.1,"MULTIPOLYGON (((-121.99727 37.15208, -121.9980..."


**2.** Print `.area` in its current CRS. Read the warning.

In [24]:
# your code here
print(santa_clara.area)

235    0.342953
dtype: float64


/var/folders/sn/xnmtvbt52xz2wj3g9sdmn31r0000gn/T/ipykernel_176/677424572.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(santa_clara.area)


**3.** Convert to EPSG:3310 and to EPSG:3857. Print each area in km².

In [25]:
# your code here
sc_3310 = santa_clara.to_crs("EPSG:3310")
sc_3857 = santa_clara.to_crs("EPSG:3857")
area_3310_km2 = sc_3310.area / 1_000_000
area_3857_km2 = sc_3857.area / 1_000_000
print("EPSG:3310 area (km²):", area_3310_km2.round(2).tolist())
print("EPSG:3857 area (km²):", area_3857_km2.round(2).tolist())

EPSG:3310 area (km²): [3377.62]
EPSG:3857 area (km²): [5337.85]


**4.** Compare with `CENSUSAREA`, the Census land area in square miles (1 sq mi = 2.59 km²).

In [26]:
# your code here
if "CENSUSAREA" in santa_clara.columns:
    census_sq_mi = pd.to_numeric(santa_clara["CENSUSAREA"], errors="coerce")
    census_km2 = census_sq_mi * 2.59
    comparison = pd.DataFrame({
        "CENSUSAREA (sq mi)": census_sq_mi,
        "CENSUSAREA converted (km²)": census_km2,
        "EPSG:3310 area (km²)": area_3310_km2,
        "EPSG:3857 area (km²)": area_3857_km2,
    }, index=santa_clara.index)
    print(comparison.round(2).to_string(index=False))
else:
    print("CENSUSAREA column not found. Available columns:", santa_clara.columns.tolist())

 CENSUSAREA (sq mi)  CENSUSAREA converted (km²)  EPSG:3310 area (km²)  EPSG:3857 area (km²)
             1290.1                     3341.36               3377.62               5337.85


**Write two sentences:** Which number would you put in a report, and why? Why is the Web Mercator area so much larger?



## Stretch: start Step 1
Open one dataset for your own project and print its CRS and size.

In [ ]:
# your project data here
